# 02 — OOP & Modern Types

**What you'll learn:**

- Classes, fields, constructors, and methods
- Encapsulation with `private` / `public` / `protected` / package-private
- `this`, constructor chaining, and method overloading
- Inheritance with `extends`, `super`, and method overriding
- Abstract classes — when a base needs to leave methods unimplemented
- Interfaces, default methods, and static methods
- Enums — fixed sets of named constants
- **Records** — one-line immutable data classes
- **Sealed types** — a closed family of subtypes the compiler enforces
- **Pattern matching for `switch`** — destructuring values inline
- Built-in annotations: `@Override`, `@Deprecated`, `@SuppressWarnings`

Notebook 01 gave us the alphabet — types, variables, control flow. This notebook gives us the sentences. Object-oriented Java is where you start *modelling* a domain rather than just computing values, and the modern additions (records, sealed types, pattern matching) make that modelling sharper and shorter than it used to be.

## Classes — the unit of code

In Java, almost everything lives inside a class. A class is a blueprint: it declares what data an object holds (**fields**) and what an object can do (**methods**). You then create objects — also called **instances** — from the class using `new`.

```text
   class blueprint                instance
   +-------------+                +-------------+
   |   Point     |   new Point()  |  x = 3      |
   |   - x: int  |  ----------->  |  y = 4      |
   |   - y: int  |                +-------------+
   +-------------+                       ^
                                         |
                                +-------------+
                                |  x = 10     |
                                |  y = 0      |
                                +-------------+
```

One class describes the shape; many instances share that shape but carry their own data.

In [ ]:
public class Point {
    // fields — the data each instance holds
    int x;
    int y;

    // constructor — how a new instance is built
    Point(int x, int y) {
        this.x = x;
        this.y = y;
    }

    // method — behaviour an instance can perform
    double distanceFromOrigin() {
        return Math.sqrt(x * x + y * y);
    }
}

// create instances
var a = new Point(3, 4);
var b = new Point(10, 0);

a.distanceFromOrigin();   // 5.0
b.distanceFromOrigin();   // 10.0

Three shapes to register from that snippet:

- **Fields** declared inside the class body, outside any method, belong to each instance.
- The **constructor** has the same name as the class and no return type. It runs once when the object is created with `new`.
- **`this`** is the implicit reference to the current instance. Use it when a parameter name shadows a field name — `this.x = x` means "the field x of this instance equals the parameter x".

## Encapsulation and visibility

Java has four access levels. You control who can see each field and method.

| Modifier | Visible from |
|---|---|
| `public` | anywhere |
| `protected` | same package + subclasses |
| *(no modifier)* | same package only (package-private) |
| `private` | same class only |

The standard practice is **private fields, public methods**. The class owns its data; the outside world interacts through methods. This is the original meaning of *encapsulation*.

In [ ]:
public class BankAccount {
    private long balance;             // private — outside code cannot touch directly

    public BankAccount(long opening) {
        this.balance = opening;
    }

    public long getBalance() {        // public accessor
        return balance;
    }

    public void deposit(long amount) {
        if (amount <= 0) throw new IllegalArgumentException("amount must be positive");
        balance += amount;
    }
}

var acct = new BankAccount(100);
acct.deposit(50);
acct.getBalance();        // 150
// acct.balance = 99999;  // COMPILE ERROR — balance is private

The compiler enforces the visibility rules. `acct.balance = 99999` would let anyone bypass the `deposit` check; making the field private makes that error impossible at compile time. Hiding data behind methods is what lets you change the internal representation later (say, switching from `long` cents to a `BigDecimal`) without breaking every caller.

## Constructors, `this`, and overloading

A class can have multiple constructors — same name, different parameter lists. One constructor can delegate to another with `this(...)`. This is how you provide convenient defaults without duplicating logic.

In [ ]:
public class Rectangle {
    private int width;
    private int height;

    // primary constructor
    public Rectangle(int width, int height) {
        this.width = width;
        this.height = height;
    }

    // convenience constructor for squares — delegates to the primary one
    public Rectangle(int side) {
        this(side, side);
    }

    public int area() { return width * height; }
}

new Rectangle(3, 5).area();   // 15
new Rectangle(4).area();      // 16  — square

**Method overloading** works the same way: same name, different parameter list. The compiler picks the right method at the call site based on the argument types. Overloading is purely a compile-time mechanism.

## Inheritance with `extends`

A class can **extend** another class to reuse its fields and methods, and to specialise its behaviour. The class you extend is the **superclass**; your class is the **subclass**.

```text
        Animal              <- superclass
         /  \
        /    \
       Dog   Cat            <- subclasses
```

A subclass inherits every non-private field and method of its superclass. It can add new ones, and it can **override** existing ones to change their behaviour.

In [ ]:
public class Animal {
    protected String name;

    public Animal(String name) {
        this.name = name;
    }

    public String speak() {
        return name + " makes a sound";
    }
}

public class Dog extends Animal {
    public Dog(String name) {
        super(name);           // call the superclass constructor
    }

    @Override
    public String speak() {    // override — replace the inherited version
        return name + " barks";
    }
}

new Animal("generic").speak();   // generic makes a sound
new Dog("rex").speak();          // rex barks

Three things to internalise:

- **`extends`** establishes the parent-child relationship. A class can extend exactly one other class (Java has single inheritance for classes).
- **`super(...)`** calls the superclass constructor. It must be the first statement in the subclass constructor.
- **`@Override`** is a built-in annotation. It tells the compiler "this method is meant to override one in the superclass" — if the signature doesn't actually match, the compiler errors out instead of silently creating a *new* method. Always use it.

## Polymorphism

Because `Dog` *is an* `Animal`, a variable of type `Animal` can hold a `Dog`. When you call `speak()` on it, the JVM dispatches to the actual runtime type's version — that's **polymorphism**.

In [ ]:
Animal a = new Dog("rex");
a.speak();   // rex barks — Dog's override runs

var animals = java.util.List.of(
    new Animal("generic"),
    new Dog("rex"),
    new Dog("buddy")
);

for (var animal : animals) {
    System.out.println(animal.speak());
}
// generic makes a sound
// rex barks
// buddy barks

You wrote the loop against the `Animal` type, but at runtime the correct `speak()` is chosen for each object. This is how frameworks like Spring let you swap implementations without changing the calling code.

## Abstract classes

Sometimes a base class can describe *part* of the shape but cannot meaningfully implement every method — only its subclasses can fill in the gaps. Declare the class **`abstract`** and leave those methods abstract too. You cannot instantiate an abstract class directly.

In [ ]:
public abstract class Shape {
    public abstract double area();    // no body — subclasses must implement

    public String describe() {        // concrete method — shared by all subclasses
        return "a shape with area " + area();
    }
}

public class Circle extends Shape {
    private final double radius;
    public Circle(double radius) { this.radius = radius; }

    @Override
    public double area() { return Math.PI * radius * radius; }
}

public class Square extends Shape {
    private final double side;
    public Square(double side) { this.side = side; }

    @Override
    public double area() { return side * side; }
}

// new Shape();        // COMPILE ERROR — abstract, can't instantiate
new Circle(2).describe();   // a shape with area 12.566...
new Square(3).describe();   // a shape with area 9.0

Abstract classes are the right tool when you have shared *state* or shared *partial implementation* across a family of types. When you only have shared *behaviour contracts*, interfaces are usually a better fit.

## Interfaces

An **interface** is a pure contract: a set of method signatures with no fields and no state. Any class can **implement** any number of interfaces, and the compiler enforces that the class provides every method the interface declares.

In [ ]:
public interface Greeter {
    String greet(String name);          // implicitly public abstract
}

public interface Loggable {
    void log(String message);
}

public class FriendlyGreeter implements Greeter, Loggable {
    @Override
    public String greet(String name) {
        return "hello, " + name;
    }

    @Override
    public void log(String message) {
        System.out.println("[INFO] " + message);
    }
}

Greeter g = new FriendlyGreeter();
g.greet("ganesh");   // hello, ganesh

Two key contrasts with abstract classes:

- A class can implement **many** interfaces but extend only **one** class.
- An interface holds no instance state — no fields (other than constants), no constructor. It is a behavioural contract, not a partial implementation.

## Default and static methods on interfaces

Since Java 8, interfaces can carry concrete methods too. A **default method** has a body and is inherited by every implementing class; the implementer can override it if needed. A **static method** belongs to the interface itself, callable like `Greeter.banner()`.

In [ ]:
public interface Greeter {
    String greet(String name);

    // default — implementers inherit this for free
    default String greetLoudly(String name) {
        return greet(name).toUpperCase();
    }

    // static — utility method on the interface itself
    static Greeter friendly() {
        return name -> "hello, " + name;   // lambda implementing greet
    }
}

var g = Greeter.friendly();
g.greet("ganesh");          // hello, ganesh
g.greetLoudly("ganesh");    // HELLO, GANESH

Default methods are how the `java.util.Collection` and `Stream` APIs were extended in Java 8 without breaking every existing implementation. They are also why an interface with a single abstract method can be implemented by a lambda — you will see plenty of that in notebook 04.

## Enums — a fixed set of constants

When a type has a small, closed set of values — days of the week, HTTP methods, payment status — an **enum** is the right shape. Each enum constant is a singleton instance of the enum type.

In [ ]:
public enum HttpMethod {
    GET, POST, PUT, DELETE, PATCH;
}

HttpMethod m = HttpMethod.POST;

// enums work great with switch
switch (m) {
    case GET    -> System.out.println("safe");
    case POST, PUT, PATCH -> System.out.println("mutating");
    case DELETE -> System.out.println("destructive");
}

// every enum has values() and valueOf()
for (var method : HttpMethod.values()) {
    System.out.println(method);
}

HttpMethod.valueOf("GET");   // HttpMethod.GET

Java enums can also carry fields, constructors, and methods — they are full classes underneath, just with a fixed and finite set of instances. This makes them strictly more powerful than the integer-flag enums of older languages.

In [ ]:
public enum HttpStatus {
    OK(200, "ok"),
    NOT_FOUND(404, "not found"),
    SERVER_ERROR(500, "server error");

    private final int code;
    private final String message;

    HttpStatus(int code, String message) {
        this.code = code;
        this.message = message;
    }

    public int code() { return code; }
    public String message() { return message; }
}

HttpStatus.NOT_FOUND.code();      // 404
HttpStatus.NOT_FOUND.message();   // not found

## Records — one-line immutable data classes

A huge fraction of classes in real codebases are **data carriers**: a fixed set of fields, a constructor that assigns them, accessors that read them, plus `equals`, `hashCode`, and `toString`. That is dozens of lines of boilerplate for a class that holds three values.

Records (Java 16) compress that to one line.

In [ ]:
public record User(String email, String name, int age) {}

var u = new User("a@b.com", "ganesh", 30);

u.email();      // a@b.com   — auto-generated accessor
u.name();       // ganesh
u.age();        // 30
u.toString();   // User[email=a@b.com, name=ganesh, age=30]

// equals and hashCode are auto-generated from the components
var u2 = new User("a@b.com", "ganesh", 30);
u.equals(u2);                       // true
u.hashCode() == u2.hashCode();      // true

What a `record` gives you for free:

- A canonical constructor with the components in declaration order
- One accessor per component (`email()`, not `getEmail()`)
- `equals` and `hashCode` derived from all components — two records with equal field values are equal
- `toString` that lists every component
- All fields are `final`, so instances are immutable

You can still add methods, override the accessors, validate in a compact constructor, and implement interfaces. What you cannot do is add mutable state — that's the whole point.

In [ ]:
public record Money(long amountCents, String currency) {

    // compact constructor — runs before the implicit field assignment
    public Money {
        if (amountCents < 0) throw new IllegalArgumentException("negative");
        if (currency == null || currency.length() != 3) {
            throw new IllegalArgumentException("currency must be 3 letters");
        }
    }

    // additional methods are fine
    public Money plus(Money other) {
        if (!currency.equals(other.currency)) {
            throw new IllegalArgumentException("currency mismatch");
        }
        return new Money(amountCents + other.amountCents, currency);
    }
}

new Money(1000, "USD").plus(new Money(500, "USD"));   // Money[amountCents=1500, currency=USD]

## Sealed types — a closed family of subtypes

A normal class or interface is open to extension by anyone — anywhere — at any time. **Sealed types** (Java 17) let you say: "these are the only types allowed to extend me." The compiler enforces it.

Sealed types pair beautifully with pattern matching: when the compiler knows the complete set of subtypes, it can verify that your `switch` covers them all.

In [ ]:
public sealed interface Shape permits Circle, Square, Triangle {}

public record Circle(double radius)                 implements Shape {}
public record Square(double side)                   implements Shape {}
public record Triangle(double base, double height)  implements Shape {}

// no other class anywhere in the world is allowed to implement Shape

## Pattern matching for `switch`

Modern `switch` can do more than match constants — it can match **types** and **destructure records** in a single expression. Combined with sealed types, the compiler will refuse to compile a `switch` that misses a case.

In [ ]:
static double area(Shape shape) {
    return switch (shape) {
        case Circle(double r)             -> Math.PI * r * r;
        case Square(double s)             -> s * s;
        case Triangle(double b, double h) -> 0.5 * b * h;
        // no default needed — Shape is sealed, the three cases are exhaustive
    };
}

area(new Circle(2));           // 12.566...
area(new Square(3));           // 9.0
area(new Triangle(4, 5));      // 10.0

Three powers stacked in that example:

- **Type patterns** — `case Circle c ->` binds `c` as a `Circle` if the runtime type matches.
- **Record deconstruction** — `case Circle(double r) ->` pulls the components out of the record into local names.
- **Exhaustiveness checking** — because `Shape` is sealed, the compiler knows the three cases cover every possible value. Adding a fourth `Shape` subtype later would force you to update this `switch` or get a compile error.

This combination of sealed types, records, and pattern matching is Java's answer to algebraic data types in Scala or Rust. It is the single biggest shift in modelling style between old Java and modern Java.

## You can also guard a pattern

A pattern can carry a **guard** — a boolean condition that must hold for the case to match. Use `when` to add one.

In [ ]:
static String classify(Shape shape) {
    return switch (shape) {
        case Circle c when c.radius() > 100 -> "huge circle";
        case Circle c                       -> "circle";
        case Square s when s.side() == 0    -> "degenerate";
        case Square s                       -> "square";
        case Triangle t                     -> "triangle";
    };
}

## Built-in annotations

An **annotation** is metadata you attach to a class, method, field, or parameter. The compiler and various libraries read annotations to change behaviour or generate code. You will use them constantly once we reach Spring — `@Component`, `@Autowired`, `@RestController` are all annotations.

Three annotations ship with the JDK and you should know them now:

- **`@Override`** — declares "this method overrides a superclass or interface method". The compiler verifies it; if the signature drifts, you get an error.
- **`@Deprecated`** — marks a class, method, or field as obsolete. Callers see a warning.
- **`@SuppressWarnings`** — tells the compiler to silence a specific warning category for the annotated element. Use sparingly.

In [ ]:
public class Calculator {

    @Deprecated(since = "2.0", forRemoval = true)
    public int oldAdd(int a, int b) {
        return a + b;
    }

    public int add(int a, int b) {
        return a + b;
    }
}

public class ScientificCalculator extends Calculator {

    @Override
    public int add(int a, int b) {
        return super.add(a, b);
    }

    @SuppressWarnings("unchecked")
    public java.util.List<String> legacyCast(Object raw) {
        return (java.util.List<String>) raw;   // would otherwise warn
    }
}

## What's next

You can now model a domain in Java with classes, inheritance, interfaces, and the modern triplet of records + sealed types + pattern matching. You also know how to read annotations, which is the doorway to Spring's whole programming model.

Notebook 03 turns to **collections and generics**: `List`, `Set`, `Map`, the immutable factories `List.of`, generic type parameters with `<T>`, and the variance rules `? extends T` and `? super T` that show up everywhere in real Java APIs.